In [1]:
from pathlib import Path
import geopandas as gpd
from shapely.geometry import LineString, Polygon, box
import rasterio
import os
import matplotlib.pyplot as plt
import folium
from branca.colormap import LinearColormap
import os
import numpy as np
from shapely.ops import transform
import pyproj

In [ ]:
from post_processing_functions import load_traffic_centers, process_all_regions, save_results

def run_flood_analysis_multi_buffer():
    # Configuration
    traffic_centers_file = r"P:\bovenregionale-stresstest-hwn\Data\Traffic_centrals\Traffic_centers.xlsx"
    #hazard_maps_base_path = r"P:\bovenregionale-stresstest-hwn\Data\Hazard_maps\Hazard_maps-in_use" #change this to add regions from analysis
    hazard_maps_base_path = r"P:\bovenregionale-stresstest-hwn\Analysis"
    output_directory = r"P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis"
    region_list = ["ARK-NZK", "Vallei en Veluwe", "Noord-Westelijke Delta", "Noord-Brabant Oost","Achterhoek"]
    buffer_distances = [5, 10, 50, 100, 200]  # meters

    all_summaries = {}

    for buffer_distance in buffer_distances:
        print(f"\n=== Running analysis for buffer distance: {buffer_distance}m ===")
        # Step 1: Load and buffer traffic centers
        gdf_buffered = load_traffic_centers(traffic_centers_file, buffer_distance)

        # Step 2: Process all regions and flood maps
        gdf_with_floods, all_results = process_all_regions(gdf_buffered, region_list, hazard_maps_base_path)

        # Step 3: Save results in a subfolder for each buffer distance
        buffer_output_dir = f"{output_directory}\\buffer_{buffer_distance}m"
        gpkg_path, excel_path = save_results(gdf_with_floods, buffer_output_dir)

        # Generate summary statistics
        summary_stats = {
            'total_traffic_centers': len(gdf_with_floods),
            'centers_with_flood_data': (gdf_with_floods['max_flood_depth'] != -9999).sum(),
            'centers_with_flooding': (gdf_with_floods['max_flood_depth'] > 0).sum(),
            'max_flood_depth_found': gdf_with_floods['max_flood_depth'].max(),
            'mean_flood_depth': gdf_with_floods[gdf_with_floods['max_flood_depth'] != -9999]['max_flood_depth'].mean(),
            'gpkg_path': gpkg_path,
            'excel_path': excel_path
        }
        all_summaries[buffer_distance] = summary_stats

        print(f"\nAnalysis complete for buffer {buffer_distance}m!")
        print(f"Processed {len(all_results)} flood maps across {len(region_list)} regions.")

    return all_summaries

summaries = run_flood_analysis_multi_buffer()


=== Running analysis for buffer distance: 5m ===
Loading traffic centers...
Loaded 6 traffic centers with 5m buffers

=== Processing region: ARK-NZK ===
Found 1 flood map files for region ARK-NZK
Processing BRST_ARK_NZK_max_wd_merge_clipNL.tif...
Error processing geometry 1: Input shapes do not overlap raster.
Error processing geometry 3: Input shapes do not overlap raster.
Error processing geometry 5: Input shapes do not overlap raster.

=== Processing region: Vallei en Veluwe ===
Found 1 flood map files for region Vallei en Veluwe
Processing samengevoegd_max_wd_merge_clipNL.tif...
Error processing geometry 0: Input shapes do not overlap raster.
Error processing geometry 1: Input shapes do not overlap raster.
Error processing geometry 2: Input shapes do not overlap raster.
Error processing geometry 4: Input shapes do not overlap raster.
Error processing geometry 5: Input shapes do not overlap raster.

=== Processing region: Noord-Westelijke Delta ===
Found 1 flood map files for regio

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('traffic_centers_with_flood_depths')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('traffic_centers_with_flood_depths')) failed: unable to open database file"



Results saved to:
  - GeoPackage: P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis\buffer_5m\traffic_centers_with_flood_depths.gpkg
  - Excel: P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis\buffer_5m\traffic_centers_with_flood_depths.xlsx

Analysis complete for buffer 5m!
Processed 4 flood maps across 4 regions.

=== Running analysis for buffer distance: 10m ===
Loading traffic centers...
Loaded 6 traffic centers with 10m buffers

=== Processing region: ARK-NZK ===
Found 1 flood map files for region ARK-NZK
Processing BRST_ARK_NZK_max_wd_merge_clipNL.tif...
Error processing geometry 1: Input shapes do not overlap raster.
Error processing geometry 3: Input shapes do not overlap raster.
Error processing geometry 5: Input shapes do not overlap raster.

=== Processing region: Vallei en Veluwe ===
Found 1 flood map files for region Vallei en Veluwe
Processing samengevoegd_max_wd_merge_clipNL.tif...
Error processing geometry 0: Input shapes do not overl

CPLE_AppDefinedError: b'sqlite3_exec(CREATE TRIGGER "trigger_delete_feature_count_traffic_centers_with_flood_depths" AFTER DELETE ON "traffic_centers_with_flood_depths" BEGIN UPDATE gpkg_ogr_contents SET feature_count = feature_count - 1 WHERE lower(table_name) = lower(\'traffic_centers_with_flood_depths\'); END;) failed: unable to open database file'

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b'sqlite3_exec(CREATE TRIGGER "trigger_delete_feature_count_traffic_centers_with_flood_depths" AFTER DELETE ON "traffic_centers_with_flood_depths" BEGIN UPDATE gpkg_ogr_contents SET feature_count = feature_count - 1 WHERE lower(table_name) = lower(\'traffic_centers_with_flood_depths\'); END;) failed: unable to open database file'



Results saved to:
  - GeoPackage: P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis\buffer_10m\traffic_centers_with_flood_depths.gpkg
  - Excel: P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis\buffer_10m\traffic_centers_with_flood_depths.xlsx

Analysis complete for buffer 10m!
Processed 4 flood maps across 4 regions.

=== Running analysis for buffer distance: 50m ===
Loading traffic centers...
Loaded 6 traffic centers with 50m buffers

=== Processing region: ARK-NZK ===
Found 1 flood map files for region ARK-NZK
Processing BRST_ARK_NZK_max_wd_merge_clipNL.tif...
Error processing geometry 1: Input shapes do not overlap raster.
Error processing geometry 3: Input shapes do not overlap raster.
Error processing geometry 5: Input shapes do not overlap raster.

=== Processing region: Vallei en Veluwe ===
Found 1 flood map files for region Vallei en Veluwe
Processing samengevoegd_max_wd_merge_clipNL.tif...
Error processing geometry 0: Input shapes do not ov

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET min_x = 89544.50866394457, min_y = 387197.1483394147, max_x = 184033.3422713417, max_y = 496916.048572405 WHERE lower(table_name) = lower('traffic_centers_with_flood_depths') AND Lower(data_type) = 'features') failed: disk I/O error"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET min_x = 89544.50866394457, min_y = 387197.1483394147, max_x = 184033.3422713417, max_y = 496916.048572405 WHERE lower(table_name) = lower('traffic_centers_with_flood_depths') AND Lower(data_type) = 'features') failed: disk I/O error"



Results saved to:
  - GeoPackage: P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis\buffer_100m\traffic_centers_with_flood_depths.gpkg
  - Excel: P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Central_Analysis\buffer_100m\traffic_centers_with_flood_depths.xlsx

Analysis complete for buffer 100m!
Processed 4 flood maps across 4 regions.

=== Running analysis for buffer distance: 200m ===
Loading traffic centers...
Loaded 6 traffic centers with 200m buffers

=== Processing region: ARK-NZK ===
Found 1 flood map files for region ARK-NZK
Processing BRST_ARK_NZK_max_wd_merge_clipNL.tif...
Error processing geometry 1: Input shapes do not overlap raster.
Error processing geometry 3: Input shapes do not overlap raster.
Error processing geometry 5: Input shapes do not overlap raster.

=== Processing region: Vallei en Veluwe ===
Found 1 flood map files for region Vallei en Veluwe
Processing samengevoegd_max_wd_merge_clipNL.tif...
Error processing geometry 0: Input shapes do n

In [ ]:
region_list = ["Noord-Westelijke Delta","Overijsselse Vecht","Limburg","Vallei en Veluwe","Achterhoek", "Brabantse Delta","Friesland","Limburg","ARK-NZK",  "Noord-Brabant Oost",
               "Rivierenland","Scheldestromen"
               ]

In [ ]:
region_list = ["ARK-NZK"]

In [ ]:
region_list = [ "Overijsselse Vecht","Limburg","Vallei en Veluwe","Achterhoek", "Brabantse Delta","Friesland","Limburg","ARK-NZK",  "Noord-Brabant Oost",
               "Rivierenland","Scheldestromen"
               ]

In [ ]:
#Linking traffic data road segments
import pandas as pd
root_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Traffic_Analysis_Data")
root_dir.mkdir(parents=True, exist_ok=True)
data = Path(r"P:\bovenregionale-stresstest-hwn\Data\Traffic_data\INWEVA_2024 (1)")
traffic_geometry_path=data.joinpath("INWEVA_2024_netwerk\INWEVA_2024_netwerk.shp") 
werkdag_traffic_csv_path = data.joinpath("inweva_werkdag_2024DEC.csv")
traffic_geometry= gpd.read_file(traffic_geometry_path, driver='SHP') 
traffic= pd.read_csv(werkdag_traffic_csv_path, sep=';') 

traffic_subset = traffic[['VBN_ID', 'AL_D_WR', 'VRPC_D_WR']]
traffic_subset['VBN_ID'] = traffic_subset['VBN_ID'].astype(str)
result = traffic_geometry.merge(traffic_subset, left_on='vbn_id', right_on='VBN_ID', how='left')
print(f"Original traffic_geometry rows: {len(traffic_geometry)}")
print(f"Result rows: {len(result)}")
print(result.columns.tolist())

# Save the result as a geopackage
# Rename the VBN_ID column to avoid conflicts
result_renamed = result.rename(columns={'VBN_ID': 'traffic_vbn_id'})

output_path = root_dir.joinpath("traffic_werkdag_2024DEC.gpkg")
if output_path.exists():  # overwrite existing file
    output_path.unlink()
result_renamed.to_file(output_path, driver='GPKG')
print(f"Saved merged data to: {output_path}")
#print(f"Saved merged data to: {output_path}")

result_renamed = result.drop(columns=['VBN_ID'])# Rename the VBN_ID column to avoid conflicts
result_exploded = result_renamed.copy()# Split nwb_ids and create duplicate rows for each value
result_exploded['nwb_ids'] = result_exploded['nwb_ids'].str.split(', ')# Split the nwb_ids column by comma and expand into separate rows
result_exploded = result_exploded.explode('nwb_ids')
result_exploded = result_exploded.reset_index(drop=True)# Reset index after exploding
result_exploded['individual_nwb_id'] = result_exploded['nwb_ids']# Create a new column with individual nwb_id values
#print(f"Original rows: {len(result_renamed)}")
#print(f"Exploded rows: {len(result_exploded)}")

### check here!!!
# Select only the required columns from traffic and create a proper copy
#AL_D_WR = Totale INWEVA intensiteit over periode 'dag' (7:00 - 19:00 uur) alle voertuigen werkdag (in mvt)
#VRPC_D_WR = Vrachtpercentage dag werkdag, Freight percentage day working day
traffic_subset_ver01 = result_exploded[['individual_nwb_id', 'AL_D_WR', 'VRPC_D_WR']].copy() 
# Convert both columns to string to ensure they match
traffic_subset_ver01['individual_nwb_id'] = traffic_subset_ver01['individual_nwb_id'].astype(str)

for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    road_path = root_dir / "Damages_Filtering_all_columns.gpkg"
    road = gpd.read_file(road_path, driver='GPKG') 
    #print(traffic.columns)
    #print(traffic_geometry.columns)
    print(road.columns)
    print(result_exploded.columns)

    # Also convert the WVK_ID column in road to string for consistent merge
    road_copy = road.copy()
    road_copy['WVK_ID'] = road_copy['WVK_ID'].astype(str)


    
    

    # ==== NEW LOGIC: reset flood metrics based on conditions ====
    fev1_cols = [c for c in road_copy.columns if c.startswith('F_EV1')]
    fev2_cols = [c for c in road_copy.columns if c.startswith('F_EV2')]

    # 1) If remove == 1 -> set ALL F_EV1* and F_EV2* columns to 0
    if 'remove' in road_copy.columns and (road_copy['remove'] == 1).any():
        mask_remove = road_copy['remove'] == 1
        if fev1_cols:
            road_copy.loc[mask_remove, fev1_cols] = 0
        if fev2_cols:
            road_copy.loc[mask_remove, fev2_cols] = 0
        print(f"Set F_EV1* and F_EV2* to 0 for {mask_remove.sum()} rows where remove == 1")

    # 2) If F_EV1_me < 0.1 -> set all F_EV2* columns to 0
    if 'F_EV1_me' in road_copy.columns and fev2_cols:
        mask_low = road_copy['F_EV1_me'] < 0.1
        if mask_low.any():
            road_copy.loc[mask_low, fev2_cols] = 0
            print(f"Set F_EV2* to 0 for {mask_low.sum()} rows where F_EV1_me < 0.1")

    # (Optional) sanity print
    # print('Flood metric columns affected:', fev1_cols + fev2_cols)


    # Dissolve by WVK_ID to get a single segment per ID keeping max F_EV2_ma
     # Dissolve by WVK_ID to get a single segment per ID keeping max F_EV2_ma and F_EV1_me
    if 'F_EV2_ma' in road_copy.columns:
        agg_dict = {}
        for col in road_copy.columns:
            if col in ['geometry', 'WVK_ID']:
                continue
            if col in ['F_EV2_ma', 'F_EV1_me', 'F_EV1_ma']:  # take maximum for these flood metrics if present
                agg_dict[col] = 'max'
            else:
                agg_dict[col] = 'first'
        road_copy = road_copy.dissolve(by='WVK_ID', aggfunc=agg_dict).reset_index()
    else:
        print("Warning: F_EV2_ma not found; dissolve skipped.")

    # Perform left join to keep all rows from road
    Road_merge = road_copy.merge(traffic_subset_ver01, left_on='WVK_ID', right_on='individual_nwb_id', how='left')

    # Display the result
    #print(f"Original road rows: {len(road)}")
    #print(f"Result rows: {len(Road_merge)}")
    #print(f"Road_merge columns: {Road_merge.columns.tolist()}")

    # Convert WVK_ID to string and remove decimal points
    road_copy['WVK_ID'] = road_copy['WVK_ID'].astype(str).str.replace('.0', '', regex=False)

    # Perform left join to keep all rows from road
    Road_merge = road_copy.merge(traffic_subset_ver01, left_on='WVK_ID', right_on='individual_nwb_id', how='left')

    # Display the result
    #print(f"Original road rows: {len(road)}")
    #print(f"Result rows: {len(Road_merge)}")
    #print(f"Road_merge columns: {Road_merge.columns.tolist()}")

    # Check data types and sample values after cleaning
    #print("\nAfter cleaning:")
    #print(f"Sample WVK_ID values from road: {road_copy['WVK_ID'].head().tolist()}")
    #print(f"Sample individual_nwb_id values from traffic: {traffic_subset_ver01['individual_nwb_id'].head().tolist()}")

    # Check for matches
    matched_records = Road_merge['individual_nwb_id'].notna().sum()
    total_records = len(Road_merge)
    match_percentage = (matched_records / total_records) * 100

    print(f"\nMatching results:")
    print(f"Total road records: {total_records}")
    print(f"Records with traffic data: {matched_records}")
    print(f"Records without traffic data: {total_records - matched_records}")
    print(f"Match percentage: {match_percentage:.2f}%")

    print(Road_merge.columns)

    Road_merge['traffic_AL_D_WR'] = Road_merge['AL_D_WR'] * Road_merge['F_EV2_ma'] 
    Road_merge['traffic_VRPC_D_WR'] = Road_merge['VRPC_D_WR'] * Road_merge['F_EV2_ma'] 

    print(Road_merge.columns)

    road_merge_output_path = root_dir.joinpath("Traffic_Analysis_ver02.gpkg")
    Road_merge.to_file(road_merge_output_path, driver='GPKG')
    print(f"Saved merged road and traffic data to: {road_merge_output_path}")



In [ ]:
region_list = ["ARK-NZK"]

In [2]:
from post_processing_functions import Thresholding_for_artefacts,Thresholding_for_artefacts,Filter_and_aggregate_flooded_segments_exposure, Filter_and_aggregate_flooded_segments_damage, calculate_overlay_percentages,get_z_height_optimized
import pandas as pd 
# add tunnels and bridge % columns to exposure and damage files and filter

data_dir = Path(r"P:\bovenregionale-stresstest-hwn\Data\Processed_data")


tunnels = data_dir.joinpath("Tunnels_filtered_by_area_2000_th.gpkg")
bridges = data_dir.joinpath("filtered_bridges.gpkg")
kunstinweg = data_dir.joinpath("kunstinweg.shp")
road_height_path = data_dir.joinpath("road_height_points.gpkg")

kunstinweg_gdf = gpd.read_file(kunstinweg)
kunstinweg_gdf['geometry'] = kunstinweg_gdf['geometry'].buffer(0.2) # to make sure lines are valid
kunstinweg_gdf.rename(columns={'OMSCHR': 'objecttekst'}, inplace=True)

kunstinweg_bridge = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'brug']
kunstinweg_tunnel = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'tunnel']


tunnels_gdf_kunstoverweg = gpd.read_file(tunnels)
bridges_gdf_kunstoverweg = gpd.read_file(bridges) 

bridges_gdf = gpd.GeoDataFrame(
    pd.concat([bridges_gdf_kunstoverweg, kunstinweg_bridge], ignore_index=True),
    crs=bridges_gdf_kunstoverweg.crs
)

tunnels_gdf = gpd.GeoDataFrame(
    pd.concat([tunnels_gdf_kunstoverweg, kunstinweg_tunnel], ignore_index=True),
    crs=tunnels_gdf_kunstoverweg.crs
)



# Define allowed values for bridges and tunnels (lowercased for case-insensitive matching)
allowed_bridges = [
    'aanbrug', 'brug', 'brug (beweegbaar)', 'brug (landbouw)', 'brug (vast)',
    'brug beton', 'brug beton in', 'brug beton over', 'brug beweegbaar',
    'brug hout in', 'brug in', 'brug in de toerit va', 'brug staal in',
    'brug vast', 'vaste brug'
]

allowed_tunnels = [
    'cervedict tunnel', 'open tunnelbak', 'tunnel', 'tunnel vlak',
    'tunnelbak', 'tunnelbak den kaat'
]

filtered_viaducts = kunstinweg_gdf[kunstinweg_gdf['objecttekst'].str.lower().str.strip() == 'viaduct']

# Convert to lowercase for case-insensitive comparison
allowed_bridges = [x.lower() for x in allowed_bridges]
allowed_tunnels = [x.lower() for x in allowed_tunnels]

# Filter bridges
filtered_gdf_brug = bridges_gdf[
    bridges_gdf['objecttekst'].str.lower().isin(allowed_bridges)
]

# Filter tunnels
filtered_gdf_tunnel_and_bridges = tunnels_gdf[
    tunnels_gdf['objecttekst'].str.lower().isin(allowed_tunnels + allowed_bridges)
]


In [13]:
# ...existing code...
region_list = ["ARK-NZK"]

for region in region_list:
    print(f"Processing region: {region} tunnels")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "Damages_Filtering_all_columns.gpkg"
    roads_ex_gdf = gpd.read_file(roads_ex)

    # Use only allowed tunnels (not bridges)
    allowed_tunnels_only = tunnels_gdf[
        tunnels_gdf['objecttekst'].str.lower().isin(allowed_tunnels)
    ].copy()

    # Drop invalid/empty geometries
    allowed_tunnels_only = allowed_tunnels_only[
        allowed_tunnels_only.geometry.notna() & ~allowed_tunnels_only.geometry.is_empty
    ].copy()

    # Ensure CRS match
    if allowed_tunnels_only.crs != roads_ex_gdf.crs:
        allowed_tunnels_only = allowed_tunnels_only.to_crs(roads_ex_gdf.crs)

    # Keep only the flag and geometry from roads, coerce to 0/1
    if 'flooded_tunnel_entrance' not in roads_ex_gdf.columns:
        raise KeyError("Column 'flooded_tunnel_entrance' not found in roads_ex_gdf")
    roads_flags = roads_ex_gdf[['flooded_tunnel_entrance', 'geometry']].copy()
    roads_flags['flooded_tunnel_entrance'] = (
        roads_flags['flooded_tunnel_entrance'].fillna(0).astype(float).gt(0).astype(int)
    )

    # Spatial join: tunnels vs roads
    joined = gpd.sjoin(
        allowed_tunnels_only,
        roads_flags,
        how='left',
        predicate='intersects'
    )

    # Aggregate per tunnel: flooded if any intersecting road has flag == 1
    flooded_by_tunnel = (
        joined.groupby(joined.index)['flooded_tunnel_entrance']
        .max()
        .reindex(allowed_tunnels_only.index)
        .fillna(0)
        .astype(int)
    )

    # Add the flooded flag (per tunnel)
    tunnels_out = allowed_tunnels_only.copy()
    tunnels_out['flooded'] = flooded_by_tunnel.values

    # Save per-tunnel result
    output_gpkg = root_dir.joinpath("allowed_tunnels_flooded.gpkg")
    if output_gpkg.exists():
        output_gpkg.unlink()
    tunnels_out.to_file(output_gpkg, driver="GPKG")
    print(f"Saved: {output_gpkg} | flooded={(tunnels_out['flooded']==1).sum()} / {len(tunnels_out)}")

    # ---- Dissolve tunnels that touch OR are within 1 m ----
    TOLERANCE_M = 1.0  # meters

    # Clean index for spatial index bookkeeping
    tunnels_out = tunnels_out.reset_index(drop=True)

    # Skip if empty
    if len(tunnels_out) == 0 or tunnels_out.geometry.is_empty.all():
        print("No tunnel features to dissolve.")
        continue

    # Build proximity connectivity using buffered queries (distance <= TOLERANCE_M)
    try:
        buffered = tunnels_out.geometry.buffer(TOLERANCE_M)
        idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')
    except Exception as e:
        print(f"Spatial index query failed: {e}")
        idx_src = idx_tgt = []

    # Union-Find for connected components
    n = len(tunnels_out)
    parent = list(range(n))
    rank = [0] * n

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra == rb:
            return
        if rank[ra] < rank[rb]:
            parent[ra] = rb
        elif rank[ra] > rank[rb]:
            parent[rb] = ra
        else:
            parent[rb] = ra
            rank[ra] += 1

    # Add edges (skip self-pairs)
    for a, b in zip(idx_src, idx_tgt):
        if a != b:
            if b < a:
                a, b = b, a
            union(int(a), int(b))

    # Component ids
    roots = [find(i) for i in range(n)]
    unique_roots = {r: i for i, r in enumerate(sorted(set(roots)))}
    tunnels_out['comp_id'] = [unique_roots[r] for r in roots]

    # Aggregate attributes: flooded=max; others=first
    agg = {c: 'first' for c in tunnels_out.columns if c not in ['geometry', 'flooded', 'comp_id']}
    agg['flooded'] = 'max'

    tunnels_out_diss = tunnels_out.dissolve(by='comp_id', aggfunc=agg).reset_index(drop=True)

    # Save dissolved result
    output_gpkg_diss = root_dir.joinpath("allowed_tunnels_flooded_dissolved.gpkg")
    if output_gpkg_diss.exists():
        output_gpkg_diss.unlink()
    tunnels_out_diss.to_file(output_gpkg_diss, driver="GPKG")
    print(
        f"Saved: {output_gpkg_diss} | groups={len(tunnels_out_diss)} | "
        f"flooded={(tunnels_out_diss['flooded']==1).sum()}"
    )
# ...existing code...

Processing region: ARK-NZK tunnels


CPLE_AppDefinedError: b'sqlite3_exec(CREATE TABLE gpkg_extensions (table_name TEXT,column_name TEXT,extension_name TEXT NOT NULL,definition TEXT NOT NULL,scope TEXT NOT NULL,CONSTRAINT ge_tce UNIQUE (table_name, column_name, extension_name))) failed: disk I/O error'

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b'sqlite3_exec(CREATE TABLE gpkg_extensions (table_name TEXT,column_name TEXT,extension_name TEXT NOT NULL,definition TEXT NOT NULL,scope TEXT NOT NULL,CONSTRAINT ge_tce UNIQUE (table_name, column_name, extension_name))) failed: disk I/O error'
C:\Users\gunaratn\AppData\Local\Temp\ipykernel_16940\2721929546.py:74: FutureWarning: The `query_bulk()` method is deprecated and will be removed in GeoPandas 1.0. You can use the `query()` method instead.
  idx_src, idx_tgt = tunnels_out.sindex.query_bulk(buffered, predicate='intersects')


Saved: P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Outputs\allowed_tunnels_flooded.gpkg | flooded=16 / 266
Saved: P:\bovenregionale-stresstest-hwn\Analysis\ARK-NZK\Outputs\allowed_tunnels_flooded_dissolved.gpkg | groups=58 | flooded=10


In [9]:
print(tunnels_out.columns)

Index(['dtb_id', 'objecttekst', 'datummutatie', 'datumopname', 'muteerder',
       'status', 'objectcode', 'datuminwin', 'inwinner', 'oudedbwaarde',
       'topcode', 'naam', 'datumaanleg', 'faunavoorziening', 'doelsoort',
       'soort', 'alias', 'niveau', 'hyperlink', 'onderhouder', 'beheerder',
       'eigenaar', 'tekening', 'diameterduiker', 'photo', 'gisobjid',
       'systeemdeel', 'beheerobject', 'element', 'bouwdeel', 'datum_vervallen',
       'se_anno_cad_data', 'lengte', 'oppervlakte', 'st_area_shape_',
       'st_length_shape_', 'gdb_geomattr_data', 'fid_2', 'gml_id',
       'nationalCo', 'localId', 'namespace', 'nationalLe', 'national_1',
       'country', 'name', 'geometry', 'DOORRIJHGT', 'INVENT_OMS', 'BEGINWDL',
       'BEGINKM', 'EINDWDL', 'EINDKM', 'KANTCODE', 'WEGNUMMER', 'FK_VELD4',
       'IZI_SIDE', 'IBN', 'flooded'],
      dtype='object')


In [ ]:
from post_processing_functions import Thresholding_for_artefacts,Aggregate_flooded_segments, calculate_overlay_percentages,get_z_height_optimized


for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    roads_ex = root_dir / "damages/HZ_damage_segmented.gpkg"
    
    #roads_ex = root_dir / "hazard_overlay_with_traffic.gpkg"
    roads_ex_gdf = gpd.read_file(roads_ex)
    #roads_dm = root_dir / "damages\HZ_damage_segmented.gpkg"
    #roads_dm_gdf = gpd.read_file(roads_dm)


    points_gdf = gpd.read_file(road_height_path)
    points_gdf = points_gdf.to_crs(roads_ex_gdf.crs)
    roads_ex_gdf['Z_height'] = get_z_height_optimized(roads_ex_gdf, points_gdf, threshold=50.0)
    #roads_dm_gdf['Z_height'] = get_z_height_optimized(roads_dm_gdf, points_gdf, threshold=50.0)

    print("Calculating tunnel and bridge percentages...")
    Roads_assets = calculate_overlay_percentages(roads_ex_gdf, filtered_gdf_brug, filtered_gdf_tunnel_and_bridges,filtered_viaducts)
    #Roads_damage = calculate_overlay_percentages(roads_dm_gdf, filtered_gdf_brug, filtered_gdf_tunnel_and_bridges,filtered_viaducts)

    #Roads_exposure.to_file(root_dir / "base_network_hazard_overlay.gpkg", driver='GPKG')
    #Roads_damage.to_file(root_dir / "damages\HZ_damage_segmented_overlay.gpkg", driver='GPKG')
    print("Applying thresholding to remove artefacts...")
    #Thresholding_for_artefacts("","Failure", Roads_exposure, root_dir)
    dataframe = Thresholding_for_artefacts("F_","Damages", Roads_assets, root_dir)

    print(dataframe.columns)
    print("Filtering and aggregating flooded segments...")
    Aggregate_flooded_segments(dataframe, root_dir,"Aggregated", dissolve_col='NETWERKSCH')
    #Filter_and_aggregate_flooded_segments_damage(Roads_assets, root_dir,"Damages", dissolve_col='NETWERKSCH')



In [ ]:
#On off ramp analysis
from pathlib import Path
import geopandas as gpd
from post_processing_functions import cluster_connected,aggregate_clusters_to_points
#region_list = ["Friesland", "Vallei en Veluwe","Noord-Westelijke Delta","Limburg"]


for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    #network_file = root_dir / "damages/HZ_damage_segmented.gpkg"
    network_file = root_dir / "Damages_Filtering_all_columns.gpkg"
    #network_file = root_dir / "hazard_overlay_with_traffic.gpkg"
    network_gdf = gpd.read_file(network_file)
    #flooded_network_gdf = gpd.read_file(flooded_roads_ex)

    ramps_gdf = network_gdf[network_gdf["BST_CODE_N"].isin(["AFR", "OPR"])]
    #fl_ramps_gdf = flooded_network_gdf[flooded_network_gdf["BST_CODE_N"].isin(["AFR", "OPR"])]


    afr_gdf = ramps_gdf[ramps_gdf["BST_CODE_N"] == "AFR"].copy()
    opr_gdf = ramps_gdf[ramps_gdf["BST_CODE_N"] == "OPR"].copy()

    #fl_afr_gdf = fl_ramps_gdf[fl_ramps_gdf["BST_CODE_N"] == "AFR"].copy()
    #fl_opr_gdf = fl_ramps_gdf[fl_ramps_gdf["BST_CODE_N"] == "OPR"].copy()

    afr_gdf_clustered = cluster_connected(afr_gdf)
    #fl_afr_gdf_clustered = cluster_connected(fl_afr_gdf)
    output_gpkg = root_dir / "afr_LineSegments.gpkg"
    afr_gdf_clustered.to_file(output_gpkg, driver="GPKG")

    
    afr_gdf_aggregated = aggregate_clusters_to_points(afr_gdf_clustered, "F_EV1_me", method="max")
    output_gpkg_aggregated_afr = root_dir / "afr_Points.gpkg"
    afr_gdf_aggregated.to_file(output_gpkg_aggregated_afr, driver="GPKG")

    #fl_afr_gdf_aggregated = aggregate_clusters_to_points(fl_afr_gdf_clustered, "F_EV1_me", method="max")
    #fl_output_gpkg_aggregated_afr = root_dir / "fl_afr_Points.gpkg"
    #fl_afr_gdf_aggregated.to_file(fl_output_gpkg_aggregated_afr, driver="GPKG")

    opr_gdf_clustered = cluster_connected(opr_gdf)
    #fl_opr_gdf_clustered = cluster_connected(fl_opr_gdf)
    output_gpkg = root_dir / "opr_LineSegments.gpkg"
    opr_gdf_clustered.to_file(output_gpkg, driver="GPKG")

    opr_gdf_aggregated = aggregate_clusters_to_points(opr_gdf_clustered, "F_EV1_me", method="max")
    output_gpkg_aggregated = root_dir / "opr_Points.gpkg"
    opr_gdf_aggregated.to_file(output_gpkg_aggregated, driver="GPKG")

    #fl_opr_gdf_aggregated = aggregate_clusters_to_points(fl_opr_gdf_clustered, "F_EV1_me", method="max")
    #fl_output_gpkg_aggregated_opr = root_dir / "fl_opr_Points.gpkg"
    #fl_opr_gdf_aggregated.to_file(fl_output_gpkg_aggregated_opr, driver="GPKG")


In [ ]:
# getting high risk schakels
#-> for Failure and Damages

for region in region_list:
    print(f"Processing region: {region} network")
    root_dir = Path(rf"P:\bovenregionale-stresstest-hwn\Analysis\{region}\Outputs")
    AG_roads_ex = root_dir / "Failure_Aggregated.gpkg"
    AG_roads_ex_gdf = gpd.read_file(AG_roads_ex)
    AG_roads_dm = root_dir / "Damages_Aggregated.gpkg"
    AG_roads_dm_gdf = gpd.read_file(AG_roads_dm)
    
    #Rsk for failure
    #which ones has the highest fraction flooded
    AG_roads_ex_gdf['fr_flooded_risk_rank'] = AG_roads_ex_gdf['fraction_flooded'].rank(ascending=False, method='min')
    #Which ones has the highest max flood depth
    AG_roads_ex_gdf['Max_flooded_risk_rank'] = AG_roads_ex_gdf['EV1_me_max'].rank(ascending=False, method='min')
    #Which ones has the highest max flood depth
    AG_roads_ex_gdf['fracton_depth'] = AG_roads_ex_gdf['EV1_me_mean'] * AG_roads_ex_gdf['fraction_flooded']
    AG_roads_ex_gdf['fracton_depth_risk_rank'] = AG_roads_ex_gdf['fracton_depth'].rank(ascending=False, method='min')
    
    
    #Risk for damages
    AG_roads_dm_gdf['Lower_dm_risk_rank'] = AG_roads_dm_gdf['lower_dam_EV1_HZ_sum'].rank(ascending=False, method='min')
    AG_roads_dm_gdf['Upper_dm_risk_rank'] = AG_roads_dm_gdf['upper_dam_EV1_HZ_sum'].rank(ascending=False, method='min')
    
    AG_roads_ex_gdf.to_file(root_dir / "Failure_Aggregated_with_risk.gpkg", driver='GPKG')
    AG_roads_dm_gdf.to_file(root_dir / "Damages_Aggregated_with_risk.gpkg", driver='GPKG')

    